In [ ]:
!pip install gdown -q

import gdown
import pandas as pd

url = f'https://drive.google.com/uc?id=1lklqBJlQyQ_hZRpWev-XNL8CTTzeyzmM'
output = 'cleaned2.csv'

gdown.download(url, output, quiet=False)
df = pd.read_csv(output)
df.head()

## Предобработка датасета

In [4]:
!pip install pymystem3 -q
from pymystem3 import Mystem

In [5]:
import nltk
import re
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

In [7]:
mystem=Mystem()
def preprocess(s):
  s = s.lower() #в нижний регистр
  s = re.sub(r'[^а-яё\s]', '', s) #убираем все кроме букв и пробелов
  s = word_tokenize(s, language='russian') #токен
  sws = set(stopwords.words('russian'))
  s = [i for i in s if i not in sws and len(i)>=2 ] #убираем стоп слова
  s= ' '.join(s)
  lems = mystem.lemmatize(s)  #->список с ' '
  lems = [i for i in lems if i and i!=' '] #убираем ' '
  return ' '.join(lems) #строка 


Installing mystem to /root/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-linux-64bit.tar.gz


In [9]:
df['textt'] = df['textt'].apply(preprocess)

In [10]:
df

,topic,textt
0,Россия,космонавт сомневаться надежность мир становить...
1,Мир,референдум вопрос самоопределение восточный ти...
2,Мир,литва засуживать участник переворот год россия...
3,Мир,киргизия вести бой граница таджикистан узбекис...
4,Мир,российский националбольшевик убирать территори...
...,...,...
11995,Спорт,болельщик спартак снимать самолет двадцать бол...
11996,Экономика,акция норильский никель обвалиться процент акц...
11997,Мир,уганда обнаруживать самый большой ископаемый о...
11998,Россия,отопление москва включать октябрь отопительный...


Делим

In [23]:
df_train, df_test = df.iloc[:n_va], df.iloc[n_va:]
X_train, y_train  = df_train['textt'] , df_train['topic']
X_test, y_test  = df_test['textt'] , df_test ['topic']

In [14]:
X_test

,textt
11000,белоруссия хранить валюта день одновременно со...
11001,российский баскетболист второй франция мужской...
11002,хэнкс жена планировать свадьба компания совлад...
11003,опек принимать решение увеличение добыча нефть...
11004,эвакуация тело погибший моряк апрк курск произ...
...,...
11995,болельщик спартак снимать самолет двадцать бол...
11996,акция норильский никель обвалиться процент акц...
11997,уганда обнаруживать самый большой ископаемый о...
11998,отопление москва включать октябрь отопительный...


Красота!

## Обучение

Теперь, когда мы предобработали датасет, нам надо сделать Tf Idf и обучить Log Reg по нему. У обоих много гиперпараметров, поэтому давайте сделаем Оптуну для подбора гиперпараметров

In [77]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
!pip install optuna

In [22]:
import optuna

In [47]:
from sklearn.pipeline import make_pipeline

def objective(trial):
  max_features = trial.suggest_int('max_features', 500, 1500)
  min_df = trial.suggest_float('min_df', 0.01, 0.05)
  max_df = trial.suggest_float( 'max_df', 0.6, 0.95)

  C = trial.suggest_float('C', 0.1, 10, log=True)
  penalty = trial.suggest_categorical('penalty', ['l2',  None])
  class_weight = trial.suggest_categorical('class_weight', ['balanced', None])

  vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df = min_df, max_df=max_df)
  model = LogisticRegression(max_iter=700, C=C, multi_class = 'multinomial', penalty=penalty, class_weight=class_weight )

  cv = TimeSeriesSplit(n_splits=3)
  scores = []

  for train_idx, val_idx in cv.split(X_train, y_train):
      X_tr = X_train.iloc[train_idx]
      X_val = X_train.iloc[val_idx]
      y_tr = y_train.iloc[train_idx]
      y_val = y_train.iloc[val_idx]

      X_tr_vec = vectorizer.fit_transform(X_tr)
      model.fit(X_tr_vec, y_tr)

      X_val_vec = vectorizer.transform(X_val)
      y_pred = model.predict(X_val_vec)

      scores.append(accuracy_score(y_val, y_pred))

  return np.mean(scores)

In [ ]:
study = optuna.create_study(direction= "maximize")
study.optimize(objective, n_trials=20)

In [61]:
print(study.best_value)
print(study.best_trial.params)

0.8630303030303031
{'max_features': 1200, 'min_df': 0.010491363931729825, 'max_df': 0.8671482762014893, 'C': 2.0994274887133852, 'penalty': 'l2', 'class_weight': 'balanced'}


In [66]:
vectorizer = TfidfVectorizer(max_features = 1200, min_df= 0.010491363931729825,  max_df= 0.8671482762014893, ngram_range=(1, 2))

X_trainv = vectorizer.fit_transform(X_train)
X_testv = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=700, multi_class = 'multinomial', C= 2.0994274887133852, penalty='l2',  class_weight = 'balanced')
model.fit(X_trainv, y_train)
y_preds = model.predict(X_testv)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [65]:
from sklearn.metrics import classification_report, accuracy_score

print(f'Accuracy {accuracy_score(y_test, y_preds)*100:.1f}%')

print(classification_report(y_test, y_preds))

Accuracy 85.3%
              precision    recall  f1-score   support

    Культура       0.91      0.86      0.88        49
         Мир       0.74      0.88      0.81       228
      Россия       0.91      0.83      0.87       501
       Спорт       0.99      0.85      0.91        78
   Экономика       0.79      0.89      0.84       144

    accuracy                           0.85      1000
   macro avg       0.87      0.86      0.86      1000
weighted avg       0.86      0.85      0.85      1000

